# Oklahoma 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Oklahoma, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note, there is no primary election dataset for Oklahoma 2008 so far.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total` , `dem_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [2]:
# OK 2008 dataset path
# PRIMARY_PATH = r""
GENERAL_PATH = r"../../data/raw/2008/OK/20081104__ok__general__president__county.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/OK/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### b. General Election Dataset

In [3]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Adair,President,NaN,R,JOHN McCAIN and SARAH PALIN,4638
1,Adair,President,NaN,D,BARACK OBAMA and JOE BIDEN,2052
2,Alfalfa,President,NaN,R,JOHN McCAIN and SARAH PALIN,2023
3,Alfalfa,President,NaN,D,BARACK OBAMA and JOE BIDEN,411
4,Atoka,President,NaN,R,JOHN McCAIN and SARAH PALIN,3511
5,Atoka,President,NaN,D,BARACK OBAMA and JOE BIDEN,1370
6,Beaver,President,NaN,R,JOHN McCAIN and SARAH PALIN,2199
7,Beaver,President,NaN,D,BARACK OBAMA and JOE BIDEN,265
8,Beckham,President,NaN,R,JOHN McCAIN and SARAH PALIN,5772
9,Beckham,President,NaN,D,BARACK OBAMA and JOE BIDEN,1625


In [4]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President    154
Name: count, dtype: int64

In [5]:
# Number of missing values in each column
general_df.isna().sum()

county         0
office         0
district     154
party          0
candidate      0
votes          0
dtype: int64

The data specifically only have values for the presidential election. Thus, we can drop the `office` column, since they all contain the same value. Also, given that there are 154 rows in `general_df` and 154 missing values in `district`, we can further drop this column.

In [6]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Adair,R,JOHN McCAIN and SARAH PALIN,4638
1,Adair,D,BARACK OBAMA and JOE BIDEN,2052
2,Alfalfa,R,JOHN McCAIN and SARAH PALIN,2023
3,Alfalfa,D,BARACK OBAMA and JOE BIDEN,411
4,Atoka,R,JOHN McCAIN and SARAH PALIN,3511
5,Atoka,D,BARACK OBAMA and JOE BIDEN,1370
6,Beaver,R,JOHN McCAIN and SARAH PALIN,2199
7,Beaver,D,BARACK OBAMA and JOE BIDEN,265
8,Beckham,R,JOHN McCAIN and SARAH PALIN,5772
9,Beckham,D,BARACK OBAMA and JOE BIDEN,1625


In [7]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
JOHN McCAIN and SARAH PALIN    77
BARACK OBAMA and JOE BIDEN     77
Name: count, dtype: int64

Now, each row’s candidate value now contains two names: presidential first, vice-presidential second, which is separated by a "and". We’ll split on the "and" and retain only the presidential name.

In [9]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
).str.upper()

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
JOHN MCCAIN     77
BARACK OBAMA    77
Name: count, dtype: int64

In [10]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
R    77
D    77
Name: count, dtype: int64

In [11]:
# Data type of each column in general_df
general_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [12]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Adair,R,JOHN MCCAIN,4638
1,Adair,D,BARACK OBAMA,2052
2,Alfalfa,R,JOHN MCCAIN,2023
3,Alfalfa,D,BARACK OBAMA,411
4,Atoka,R,JOHN MCCAIN,3511
5,Atoka,D,BARACK OBAMA,1370
6,Beaver,R,JOHN MCCAIN,2199
7,Beaver,D,BARACK OBAMA,265
8,Beckham,R,JOHN MCCAIN,5772
9,Beckham,D,BARACK OBAMA,1625


In [13]:
# Shape after preprocessing
general_df.shape

(154, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [15]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "D"  : "dem",                     
                "R"  : "rep"
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [16]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [17]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [18]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_rep_MCCAIN
0,Adair,2052,4638
1,Alfalfa,411,2023
2,Atoka,1370,3511
3,Beaver,265,2199
4,Beckham,1625,5772
5,Blaine,1011,3101
6,Bryan,4426,9307
7,Caddo,3404,6413
8,Canadian,11426,36428
9,Carter,5603,13241


In [19]:
# General dataframe shape after pivot
general_pivot.shape

(77, 3)

## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns

In [20]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]

general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0

In [21]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned general dataframe:")
general_pivot.columns

Final columns in the cleaned general dataframe:


Index(['county', 'gen_dem_OBAMA', 'gen_rep_MCCAIN', 'rep_general_total',
       'dem_general_total'],
      dtype='object')

In [22]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_rep_MCCAIN,rep_general_total,dem_general_total
0,Adair,2052,4638,4638,2052
1,Alfalfa,411,2023,2023,411
2,Atoka,1370,3511,3511,1370
3,Beaver,265,2199,2199,265
4,Beckham,1625,5772,5772,1625
5,Blaine,1011,3101,3101,1011
6,Bryan,4426,9307,9307,4426
7,Caddo,3404,6413,6413,3404
8,Canadian,11426,36428,36428,11426
9,Carter,5603,13241,13241,5603


Now, we save the cleaned dataframe into the processed directory.

In [23]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "OK.csv", index=False)